# 6장. 쌓을 수 있는 블록: Transformer 블록

어텐션에 MLP, LayerNorm, 잔차 연결을 더해 **여러 층 쌓을 수 있는 블록**을 만들어요. 이 장은 **GPU가 필요 없어요.** (런타임 유형: CPU)


### 준비
5장의 결과물 `MultiHead`를 패키지에서 불러옵니다.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import torch
from torch import nn
from modu_llm import Vocab, GPTConfig, Embeddings, MultiHead

torch.manual_seed(42)
vocab = Vocab.build()
cfg = GPTConfig(vocab_size=len(vocab), block_size=16, n_embd=16, n_head=4)
emb = Embeddings(cfg)
idx = torch.tensor(vocab.encode("토끼가 숲에 가요"))
x = emb(idx)
print("x 모양:", x.shape)


## 6.1 글자마다 정보 가공하기: MLP

어텐션은 글자들 사이에서 정보를 **모으는** 일을 합니다. 모은 정보를 각 글자 안에서 **정리하는** 일은 MLP가 맡아요.
벡터를 4배로 늘렸다가, GELU로 꺾어 주고, 다시 원래 길이로 줄입니다.

**핵심 코드**: `nn.Sequential(Linear, GELU, Linear)`


In [ ]:
mlp = nn.Sequential(
    nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
    nn.GELU(),
    nn.Linear(4 * cfg.n_embd, cfg.n_embd),
)
out = mlp(x)
print("입력:", x.shape)
print("출력:", out.shape)


GELU가 숫자를 어떻게 꺾는지 몇 개의 숫자로 확인해 봅니다.


In [ ]:
gelu = nn.GELU()
nums = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
print("입력:", nums)
print("GELU:", gelu(nums))


MLP는 **글자마다 따로** 계산합니다. 첫 글자만 넣은 결과와, 9글자를 모두 넣은 결과의 첫 글자 부분을 비교해 보세요.


In [ ]:
first_alone = mlp(x[:1])
first_in_all = out[:1]
same = torch.allclose(first_alone, first_in_all, atol=1e-6)
print("첫 글자만 넣은 결과와 같은가요?", same)


## 6.2 값을 고르게 맞추기: LayerNorm

층을 여러 번 지나면 벡터 값이 너무 커지거나 작아져 학습이 불안정해져요. LayerNorm은 글자마다 벡터 값의 **평균을 0, 퍼진 정도를 1**로 맞춥니다.

**핵심 코드**: `nn.LayerNorm(n_embd)`


In [ ]:
ln = nn.LayerNorm(cfg.n_embd)
big = x * 5 + 3        # 일부러 값을 크게 만든 입력
after = ln(big)

v1 = big[0]            # 첫 글자 '토'의 벡터
v2 = after[0]
mean1 = v1.mean().item()
std1 = v1.std().item()
mean2 = v2.mean().item()
std2 = v2.std().item()
print(f"LayerNorm 전 | 평균 {mean1:.2f}, 표준편차 {std1:.2f}")
print(f"LayerNorm 후 | 평균 {mean2:.2f}, 표준편차 {std2:.2f}")


## 6.3 원래 값을 더해 주기: 잔차 연결

어텐션이나 MLP의 결과로 벡터를 **바꿔 치우지 않고, 원래 값에 더해요**. 각 층은 "원래 정보에 얹을 보탬"만 배우면 됩니다.

**핵심 코드**: `y = x + attn(ln(x))`


In [ ]:
attn = MultiHead(cfg)
y = x + attn(ln(x))
print("잔차 연결 뒤 모양:", y.shape)


## 6.4 블록 완성하고 여러 층 쌓기

LayerNorm → 어텐션 → 더하기 → LayerNorm → MLP → 더하기. 이것이 Transformer 블록 하나예요.
입력과 출력의 모양이 같으므로 여러 층을 그대로 쌓을 수 있습니다.

**핵심 코드**: `x = x + self.attn(self.ln1(x))`


In [ ]:
class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = MultiHead(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

blocks = nn.ModuleList()
for i in range(3):
    blocks.append(Block(cfg))

h = x
for block in blocks:
    h = block(h)
print("블록 3개 통과:", x.shape, "→", h.shape)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

블록을 하나씩 지날 때마다 벡터가 얼마나 달라지는지 재어 보세요. 잔차 연결 덕분에 한 번에 크게 바뀌지 않고 조금씩 보태집니다.


In [ ]:
h = x
for i, block in enumerate(blocks):
    new = block(h)
    change = (new - h).norm() / h.norm()
    ratio = change.item()
    print(f"블록 {i + 1}: 바뀐 정도 {ratio:.2f}")
    h = new


## 실제 LLM에서는

- 실제 LLM도 이 블록을 수십~백여 층 쌓습니다. 구조는 거의 같고 세부만 달라요.
- 예를 들어 LayerNorm 대신 더 간단한 RMSNorm을, GELU 대신 SwiGLU 같은 활성화 함수를 많이 씁니다.
